# Grille 87_0

Visualisation longitude/latitude avec slack node, loads, generateurs, numeros des nodes, numeros des edges et chargement des lignes apres power flow.

In [1]:
import re
import pandas as pd
import pandapower as pp
import plotly.graph_objects as go
import plotly.io as pio
from plotly.colors import sample_colorscale

pio.renderers.default = "vscode"

grid_file = "/Users/jadlostan/Documents/GitHub/Swiss-PDGs/grids/pandapower_data/MV/87_0_grid.xlsx"
nodes_file = "nodes_valais.csv"
grid_id = "87_0"

net = pp.from_excel(grid_file)
pp.runpp(net)

nodes = pd.read_csv(nodes_file)

def id_to_str(value):
    return str(int(value)) if float(value).is_integer() else str(value)

def node_id_from_bus_name(name):
    match = re.search(r"_node_(.+)$", str(name))
    return match.group(1) if match else None

nodes_grid = nodes[nodes["grid_id"] == grid_id].dropna(subset=["osmid", "longitude", "latitude"]).copy()
nodes_grid["node_id"] = nodes_grid["osmid"].apply(id_to_str)
node_lookup = nodes_grid.drop_duplicates("node_id").set_index("node_id")[["longitude", "latitude"]]

bus_plot = net.bus.copy()
bus_plot["bus_index"] = bus_plot.index
bus_plot["node_id"] = bus_plot["name"].apply(node_id_from_bus_name)
bus_plot = bus_plot.join(node_lookup, on="node_id")
bus_plot = bus_plot.dropna(subset=["longitude", "latitude"]).set_index("bus_index")

load_by_bus = net.load[net.load["in_service"]].groupby("bus")["p_mw"].sum()
gen_by_bus = pd.Series(dtype=float)
if len(net.sgen):
    gen_by_bus = pd.concat([gen_by_bus, net.sgen[net.sgen["in_service"]].groupby("bus")["p_mw"].sum()])
if len(net.gen):
    gen_by_bus = pd.concat([gen_by_bus, net.gen[net.gen["in_service"]].groupby("bus")["p_mw"].sum()])
if len(gen_by_bus):
    gen_by_bus = gen_by_bus.groupby(level=0).sum()

slack_buses = set(net.ext_grid[net.ext_grid["in_service"]]["bus"].astype(int))
ext_grid_by_bus = {}
for ext_idx, ext in net.ext_grid[net.ext_grid["in_service"]].iterrows():
    ext_grid_by_bus[int(ext["bus"])] = (
        float(net.res_ext_grid.loc[ext_idx, "p_mw"]),
        float(net.res_ext_grid.loc[ext_idx, "q_mvar"]),
    )
max_load = max(load_by_bus.max(), 1e-9) if len(load_by_bus) else 1
max_gen = max(gen_by_bus.max(), 1e-9) if len(gen_by_bus) else 1

fig = go.Figure()
max_loading = max(float(net.res_line["loading_percent"].max()), 1)

for line_idx, line in net.line.iterrows():
    from_bus = int(line["from_bus"])
    to_bus = int(line["to_bus"])
    if from_bus not in bus_plot.index or to_bus not in bus_plot.index:
        continue

    x = [bus_plot.loc[from_bus, "longitude"], bus_plot.loc[to_bus, "longitude"]]
    y = [bus_plot.loc[from_bus, "latitude"], bus_plot.loc[to_bus, "latitude"]]
    loading = float(net.res_line.loc[line_idx, "loading_percent"])
    line_color = sample_colorscale("Viridis", loading / max_loading)[0]
    x_mid = (x[0] + x[1]) / 2
    y_mid = (y[0] + y[1]) / 2

    fig.add_trace(
        go.Scatter(
            x=[x[0], x_mid, x[1]],
            y=[y[0], y_mid, y[1]],
            mode="lines+text",
            text=[None, str(line_idx), None],
            textposition="middle center",
            line=dict(color=line_color, width=3),
            customdata=[loading, loading],
            hovertemplate=f"edge {line_idx}<br>loading: {loading:.2f}%<extra></extra>",
            showlegend=False,
        )
    )

# Invisible colorbar trace for the line loading scale.
fig.add_trace(
    go.Scatter(
        x=[None],
        y=[None],
        mode="markers",
        marker=dict(
            colorscale="Viridis",
            cmin=0,
            cmax=max_loading,
            color=[0],
            colorbar=dict(
                title="Loading line (%)",
                x=1.03,
                y=0.50,
                len=0.72,
                thickness=18,
            ),
            showscale=True,
        ),
        hoverinfo="skip",
        showlegend=False,
    )
)

plain = []
loads = []
gens = []
slacks = []

for bus_idx, row in bus_plot.iterrows():
    load = float(load_by_bus.get(bus_idx, 0.0))
    gen = float(gen_by_bus.get(bus_idx, 0.0)) if len(gen_by_bus) else 0.0
    ext_p, ext_q = ext_grid_by_bus.get(bus_idx, (0.0, 0.0))
    item = dict(bus_idx=bus_idx, lon=row["longitude"], lat=row["latitude"], load=load, gen=gen, ext_p=ext_p, ext_q=ext_q)
    if bus_idx in slack_buses:
        slacks.append(item)
    elif gen > 0:
        gens.append(item)
    elif load > 0:
        loads.append(item)
    else:
        plain.append(item)

def add_bus_trace(items, name, color, symbol, size_fn, line_color="white", line_width=1):
    if not items:
        return
    fig.add_trace(
        go.Scatter(
            x=[item["lon"] for item in items],
            y=[item["lat"] for item in items],
            mode="markers+text",
            text=[f"{item['bus_idx']} slack" if name.startswith("Slack") else str(item["bus_idx"]) for item in items],
            textposition="bottom center" if name.startswith("Slack") else "top center",
            marker=dict(
                symbol=symbol,
                size=[size_fn(item) for item in items],
                color=color,
                line=dict(color=line_color, width=line_width),
            ),
            customdata=[[item["bus_idx"], item["load"], item["gen"], item["ext_p"], item["ext_q"]] for item in items],
            hovertemplate=(
                "bus %{customdata[0]}<br>"
                "load: %{customdata[1]:.4f} MW<br>"
                "gen/sgen: %{customdata[2]:.4f} MW<br>"
                "ext_grid P: %{customdata[3]:.4f} MW<br>"
                "ext_grid Q: %{customdata[4]:.4f} Mvar"
                "<extra></extra>"
            ),
            name=name,
        )
    )

add_bus_trace(plain, "No load/gen", "lightgray", "circle", lambda item: 8, line_color="dimgray", line_width=1)
add_bus_trace(loads, "Load", "red", "circle", lambda item: 8 + 28 * item["load"] / max_load, line_color="white", line_width=1)
add_bus_trace(gens, "Generator", "green", "circle", lambda item: 12 + 28 * item["gen"] / max_gen, line_color="white", line_width=1)
add_bus_trace(slacks, "Slack / ext_grid", "gold", "star", lambda item: 46, line_color="black", line_width=3)

fig.update_layout(
    title="Grille 87_0 - power flow interactif",
    width=1050,
    height=800,
    xaxis_title="Longitude",
    yaxis_title="Latitude",
    dragmode="zoom",
    hovermode="closest",
    margin=dict(l=70, r=170, t=70, b=60),
    legend=dict(
        title="Type de node",
        x=0.01,
        y=0.99,
        xanchor="left",
        yanchor="top",
        bgcolor="rgba(255,255,255,0.88)",
        bordercolor="rgba(0,0,0,0.25)",
        borderwidth=1,
    ),
)
fig.update_yaxes(scaleanchor="x", scaleratio=1)
fig.show()

print("Power flow converged:", net.converged)
print("Max line loading (%):", net.res_line["loading_percent"].max())


Power flow converged: True
Max line loading (%): 41.802918120922946


In [2]:
import pandapower as pp

grid_file = "/Users/jadlostan/Documents/GitHub/Swiss-PDGs/grids/pandapower_data/MV/87_0_grid.xlsx"

net = pp.from_excel(grid_file)
pp.runpp(net)

print("Converged:", net.converged)
print("Bus:", len(net.bus))
print("Lines:", len(net.line))
print("Loads:", len(net.load))
print("External grids:", len(net.ext_grid))

display(net.res_bus.head())
display(net.res_line.head())
display(net.res_trafo.head())
display(net.res_ext_grid.head())
display(net.res_load.head())


Converged: True
Bus: 77
Lines: 75
Loads: 41
External grids: 1


,vm_pu,va_degree,p_mw,q_mvar
0,0.992406,-0.054259,0.362619,0.175624
1,0.992456,-0.054641,0.069308,0.033567
2,0.992389,-0.054104,0.117226,0.056775
3,0.992280,-0.055448,0.000000,0.000000
4,0.992249,-0.055175,0.000000,0.000000


,p_from_mw,q_from_mvar,p_to_mw,q_to_mvar,pl_mw,ql_mvar,i_from_ka,i_to_ka,i_ka,vm_from_pu,va_from_degree,vm_to_pu,va_to_degree,loading_percent
0,-1.286471,-0.575416,1.286539,0.574719,6.835302e-05,-0.000696,0.040994,0.040986,0.040994,0.992406,-0.054259,0.992456,-0.054641,18.633698
1,0.217462,0.103381,-0.217458,-0.104806,3.982167e-06,-0.001425,0.007004,0.007022,0.007022,0.992406,-0.054259,0.992389,-0.054104,3.191796
2,0.706391,0.296410,-0.706307,-0.296469,8.339855e-05,-0.000059,0.022283,0.022285,0.022285,0.992406,-0.054259,0.992280,-0.055448,10.611759
3,-1.355847,-0.608287,1.355883,0.607958,3.602230e-05,-0.000329,0.043225,0.043221,0.043225,0.992456,-0.054641,0.992481,-0.054833,19.647529
4,0.100232,0.048031,-0.100232,-0.048275,1.451955e-07,-0.000244,0.003233,0.003236,0.003236,0.992389,-0.054104,0.992388,-0.054091,1.470993


,p_hv_mw,q_hv_mvar,p_lv_mw,q_lv_mvar,pl_mw,ql_mvar,i_hv_ka,i_lv_ka,vm_hv_pu,va_hv_degree,vm_lv_pu,va_lv_degree,loading_percent
0,1.230640e-14,-7.105643e-15,0.013999,0.0105,0.013999,0.0105,7.458764e-17,0.000505,0.999974,149.998125,1.0,0.0,0.069998


,p_mw,q_mvar
0,8.9332,3.76081


,p_mw,q_mvar
0,0.362619,0.175624
1,0.069308,0.033567
2,0.117226,0.056775
3,0.070529,0.034159
4,0.149158,0.072241


# Simulation BelalpSolar avec pvlib

Donnees utilisees depuis alpine-pv.ch/belalpsolar : latitude 46.39776, longitude 7.97738, altitude 2650 m, puissance DC 5.7 MW, production annuelle 12 GWh, rendement specifique 2105 kWh/kWp, part hiver 43%.

Le site ne donne pas l'inclinaison ni l'orientation des modules. Elles sont donc posees comme hypotheses modifiables dans le code. Le profil horaire est calcule avec pvlib, puis calibre pour atteindre les 12 GWh/an annonces.

In [3]:
import pandas as pd
import pvlib
import plotly.graph_objects as go
import plotly.io as pio
from plotly.subplots import make_subplots
from pvlib.location import Location

pio.renderers.default = "vscode"

# Donnees BelalpSolar depuis https://alpine-pv.ch/belalpsolar/
latitude = 46.39776
longitude = 7.97738
altitude_m = 2650
p_dc_mw = 5.7
annual_yield_gwh_target = 12.0
specific_yield_kwh_per_kwp = 2105
winter_share = 0.43

# Hypotheses modifiables: le site ne donne pas l'inclinaison/orientation.
surface_tilt = 60      # degres, hypothese alpine orientee hiver
surface_azimuth = 180  # 180 = sud
albedo = 0.35          # neige/sol clair en altitude; a ajuster selon scenario

location = Location(
    latitude=latitude,
    longitude=longitude,
    tz="Europe/Zurich",
    altitude=altitude_m,
    name="BelalpSolar",
)

times = pd.date_range("2025-01-01 00:00", "2025-12-31 23:00", freq="h", tz=location.tz)

solar_position = location.get_solarposition(times)
clear_sky = location.get_clearsky(times, model="ineichen")

poa = pvlib.irradiance.get_total_irradiance(
    surface_tilt=surface_tilt,
    surface_azimuth=surface_azimuth,
    solar_zenith=solar_position["apparent_zenith"],
    solar_azimuth=solar_position["azimuth"],
    dni=clear_sky["dni"],
    ghi=clear_sky["ghi"],
    dhi=clear_sky["dhi"],
    albedo=albedo,
)

cell_temperature = pvlib.temperature.faiman(
    poa_global=poa["poa_global"],
    temp_air=5,
    wind_speed=2,
)

pdc_kw_raw = pvlib.pvsystem.pvwatts_dc(
    effective_irradiance=poa["poa_global"],
    temp_cell=cell_temperature,
    pdc0=p_dc_mw * 1000,
    gamma_pdc=-0.0035,
)

pac_kw_raw = pvlib.inverter.pvwatts(
    pdc=pdc_kw_raw,
    pdc0=p_dc_mw * 1000,
    eta_inv_nom=0.96,
).clip(lower=0)

# Calibration sur la production annuelle annoncee par le projet.
annual_target_kwh = annual_yield_gwh_target * 1_000_000
scale_factor = annual_target_kwh / pac_kw_raw.sum()
pac_kw = pac_kw_raw * scale_factor
energy_kwh = pac_kw

monthly_energy_mwh = energy_kwh.resample("ME").sum() / 1000
annual_energy_gwh = energy_kwh.sum() / 1_000_000
winter_energy_gwh = energy_kwh[energy_kwh.index.month.isin([10, 11, 12, 1, 2, 3])].sum() / 1_000_000
winter_percent = 100 * winter_energy_gwh / annual_energy_gwh

print(f"Production annuelle simulee calibree: {annual_energy_gwh:.2f} GWh")
print(f"Production annuelle annoncee: {annual_yield_gwh_target:.2f} GWh")
print(f"Rendement specifique annonce: {specific_yield_kwh_per_kwp} kWh/kWp")
print(f"Production hiver simulee Oct-Mar: {winter_energy_gwh:.2f} GWh ({winter_percent:.1f}%)")
print(f"Part hiver annoncee: {winter_share:.0%}")
print(f"Facteur de calibration applique au profil pvlib: {scale_factor:.3f}")

fig = make_subplots(
    rows=2,
    cols=1,
    shared_xaxes=False,
    vertical_spacing=0.12,
    subplot_titles=(
        "BelalpSolar - production horaire simulee sur une annee",
        "BelalpSolar - production mensuelle simulee",
    ),
)

fig.add_trace(
    go.Scatter(
        x=energy_kwh.index,
        y=energy_kwh / 1000,
        mode="lines",
        line=dict(color="royalblue", width=1),
        name="Production horaire",
        hovertemplate="%{x}<br>%{y:.3f} MWh<extra></extra>",
    ),
    row=1,
    col=1,
)

fig.add_trace(
    go.Bar(
        x=monthly_energy_mwh.index.strftime("%b"),
        y=monthly_energy_mwh.values,
        marker_color="orange",
        name="Production mensuelle",
        hovertemplate="%{x}<br>%{y:.1f} MWh<extra></extra>",
    ),
    row=2,
    col=1,
)

fig.update_yaxes(title_text="Production horaire (MWh)", row=1, col=1)
fig.update_yaxes(title_text="Production mensuelle (MWh)", row=2, col=1)
fig.update_layout(
    width=1100,
    height=800,
    dragmode="zoom",
    hovermode="x unified",
    showlegend=False,
    title="BelalpSolar - simulation pvlib zoomable",
)
fig.show()


Production annuelle simulee calibree: 12.00 GWh
Production annuelle annoncee: 12.00 GWh
Rendement specifique annonce: 2105 kWh/kWp
Production hiver simulee Oct-Mar: 5.71 GWh (47.6%)
Part hiver annoncee: 43%
Facteur de calibration applique au profil pvlib: 0.803


# Grille 87_0 avec BelalpSolar raccorde par une ligne pandapower

Cette cellule cree un nouveau bus a la position du projet, ajoute un generateur PV, raccorde ce bus au bus 87_0 le plus proche avec les memes parametres de cable que la ligne existante connectee a ce bus, puis relance le power flow.

In [5]:
import re
import math
import pandas as pd
import pandapower as pp
import plotly.graph_objects as go
import plotly.io as pio
from plotly.colors import sample_colorscale

pio.renderers.default = "vscode"

grid_file = "/Users/jadlostan/Documents/GitHub/Swiss-PDGs/grids/pandapower_data/MV/87_0_grid.xlsx"
nodes_file = "nodes_valais.csv"
grid_id = "87_0"

pv_name = "BelalpSolar PV"
pv_latitude = 46.39776
pv_longitude = 7.97738
pv_p_mw = 5.7
pv_q_mvar = 0.0

net = pp.from_excel(grid_file)
nodes = pd.read_csv(nodes_file)

def id_to_str(value):
    return str(int(value)) if float(value).is_integer() else str(value)

def node_id_from_bus_name(name):
    match = re.search(r"_node_(.+)$", str(name))
    return match.group(1) if match else None

def haversine_km(lon1, lat1, lon2, lat2):
    radius_km = 6371.0
    lon1, lat1, lon2, lat2 = map(math.radians, [lon1, lat1, lon2, lat2])
    dlon = lon2 - lon1
    dlat = lat2 - lat1
    a = math.sin(dlat / 2) ** 2 + math.cos(lat1) * math.cos(lat2) * math.sin(dlon / 2) ** 2
    return 2 * radius_km * math.asin(math.sqrt(a))

nodes_grid = nodes[nodes["grid_id"] == grid_id].dropna(subset=["osmid", "longitude", "latitude"]).copy()
nodes_grid["node_id"] = nodes_grid["osmid"].apply(id_to_str)
node_lookup = nodes_grid.drop_duplicates("node_id").set_index("node_id")[["longitude", "latitude"]]

bus_plot = net.bus.copy()
bus_plot["bus_index"] = bus_plot.index
bus_plot["node_id"] = bus_plot["name"].apply(node_id_from_bus_name)
bus_plot = bus_plot.join(node_lookup, on="node_id")
bus_plot = bus_plot.dropna(subset=["longitude", "latitude"]).set_index("bus_index")

# Bus de raccordement: bus existant le plus proche de la position BelalpSolar.
distance2 = (bus_plot["longitude"] - pv_longitude) ** 2 + (bus_plot["latitude"] - pv_latitude) ** 2
nearest_bus = int(distance2.idxmin())
nearest_lon = float(bus_plot.loc[nearest_bus, "longitude"])
nearest_lat = float(bus_plot.loc[nearest_bus, "latitude"])
connection_length_km = haversine_km(nearest_lon, nearest_lat, pv_longitude, pv_latitude)

# Cable: reprendre la ligne existante connectee au bus le plus proche. Sinon, prendre le type le plus frequent.
attached_lines = net.line[(net.line["from_bus"] == nearest_bus) | (net.line["to_bus"] == nearest_bus)]
if len(attached_lines):
    cable = attached_lines.iloc[0]
else:
    cable = net.line.loc[net.line[["r_ohm_per_km", "x_ohm_per_km", "c_nf_per_km", "max_i_ka", "type"]].value_counts().index[0]]

pv_bus = pp.create_bus(
    net,
    vn_kv=float(net.bus.loc[nearest_bus, "vn_kv"]),
    name="bus_MV_87_0_BelalpSolar",
    type="n",
)

pp.create_line_from_parameters(
    net,
    from_bus=nearest_bus,
    to_bus=pv_bus,
    length_km=connection_length_km,
    r_ohm_per_km=float(cable["r_ohm_per_km"]),
    x_ohm_per_km=float(cable["x_ohm_per_km"]),
    c_nf_per_km=float(cable["c_nf_per_km"]),
    max_i_ka=float(cable["max_i_ka"]),
    type=str(cable["type"]),
    name="line_87_0_to_BelalpSolar",
)

pp.create_sgen(
    net,
    bus=pv_bus,
    p_mw=pv_p_mw,
    q_mvar=pv_q_mvar,
    name=pv_name,
    type="PV",
)

pp.runpp(net)

# Ajouter les coordonnees du nouveau bus PV pour le plot.
bus_plot = pd.concat([
    bus_plot,
    pd.DataFrame(
        {"longitude": [pv_longitude], "latitude": [pv_latitude], "name": ["bus_MV_87_0_BelalpSolar"], "node_id": ["BelalpSolar"]},
        index=[pv_bus],
    ),
])

load_by_bus = net.load[net.load["in_service"]].groupby("bus")["p_mw"].sum()
sgen_by_bus = net.sgen[net.sgen["in_service"]].groupby("bus")["p_mw"].sum() if len(net.sgen) else pd.Series(dtype=float)
slack_buses = set(net.ext_grid[net.ext_grid["in_service"]]["bus"].astype(int))
max_load = max(load_by_bus.max(), 1e-9) if len(load_by_bus) else 1
max_sgen = max(sgen_by_bus.max(), 1e-9) if len(sgen_by_bus) else 1
max_loading = max(float(net.res_line["loading_percent"].max()), 1)

fig = go.Figure()

for line_idx, line in net.line.iterrows():
    from_bus = int(line["from_bus"])
    to_bus = int(line["to_bus"])
    if from_bus not in bus_plot.index or to_bus not in bus_plot.index:
        continue

    x0, y0 = bus_plot.loc[from_bus, ["longitude", "latitude"]]
    x1, y1 = bus_plot.loc[to_bus, ["longitude", "latitude"]]
    x_mid = (x0 + x1) / 2
    y_mid = (y0 + y1) / 2
    loading = float(net.res_line.loc[line_idx, "loading_percent"])
    line_color = sample_colorscale("Viridis", loading / max_loading)[0]

    fig.add_trace(
        go.Scatter(
            x=[x0, x_mid, x1],
            y=[y0, y_mid, y1],
            mode="lines+text",
            text=[None, str(line_idx), None],
            textposition="middle center",
            line=dict(color=line_color, width=4 if to_bus == pv_bus or from_bus == pv_bus else 3),
            hovertemplate=f"edge {line_idx}<br>loading: {loading:.2f}%<br>length: {float(line['length_km']):.3f} km<extra></extra>",
            showlegend=False,
        )
    )

fig.add_trace(
    go.Scatter(
        x=[None],
        y=[None],
        mode="markers",
        marker=dict(
            colorscale="Viridis",
            cmin=0,
            cmax=max_loading,
            color=[0],
            colorbar=dict(title="Loading line (%)", x=1.03, y=0.50, len=0.72, thickness=18),
            showscale=True,
        ),
        hoverinfo="skip",
        showlegend=False,
    )
)

plain, loads, slacks, pvs = [], [], [], []
for bus_idx, row in bus_plot.iterrows():
    load = float(load_by_bus.get(bus_idx, 0.0))
    sgen = float(sgen_by_bus.get(bus_idx, 0.0)) if len(sgen_by_bus) else 0.0
    item = dict(bus_idx=bus_idx, lon=row["longitude"], lat=row["latitude"], load=load, sgen=sgen)
    if bus_idx == pv_bus:
        pvs.append(item)
    elif bus_idx in slack_buses:
        slacks.append(item)
    elif load > 0:
        loads.append(item)
    else:
        plain.append(item)

def add_bus_trace(items, name, color, symbol, size_fn, line_color="white", line_width=1):
    if not items:
        return
    fig.add_trace(
        go.Scatter(
            x=[item["lon"] for item in items],
            y=[item["lat"] for item in items],
            mode="markers+text",
            text=["BelalpSolar PV" if name.startswith("Belalp") else (f"{item['bus_idx']} slack" if name.startswith("Slack") else str(item["bus_idx"])) for item in items],
            textposition="top center",
            marker=dict(
                symbol=symbol,
                size=[size_fn(item) for item in items],
                color=color,
                line=dict(color=line_color, width=line_width),
            ),
            customdata=[[item["bus_idx"], item["load"], item["sgen"]] for item in items],
            hovertemplate="bus %{customdata[0]}<br>load: %{customdata[1]:.4f} MW<br>sgen: %{customdata[2]:.4f} MW<extra></extra>",
            name=name,
        )
    )

add_bus_trace(plain, "No load", "lightgray", "circle", lambda item: 8, line_color="dimgray", line_width=1)
add_bus_trace(loads, "Load", "red", "circle", lambda item: 8 + 28 * item["load"] / max_load)
add_bus_trace(slacks, "Slack / ext_grid", "gold", "star", lambda item: 46, line_color="black", line_width=3)
add_bus_trace(pvs, "BelalpSolar PV", "limegreen", "diamond", lambda item: 32, line_color="darkgreen", line_width=2)

fig.update_layout(
    title="Grille 87_0 avec BelalpSolar raccorde par cable - power flow interactif",
    width=1150,
    height=850,
    xaxis_title="Longitude",
    yaxis_title="Latitude",
    dragmode="zoom",
    hovermode="closest",
    margin=dict(l=70, r=170, t=70, b=60),
    legend=dict(
        title="Elements",
        x=0.01,
        y=0.99,
        xanchor="left",
        yanchor="top",
        bgcolor="rgba(255,255,255,0.88)",
        bordercolor="rgba(0,0,0,0.25)",
        borderwidth=1,
    ),
)
fig.update_yaxes(scaleanchor="x", scaleratio=1)
fig.show()

print("Power flow converged:", net.converged)
print(f"BelalpSolar PV: {pv_p_mw:.2f} MW injectes au nouveau bus {pv_bus}")
print(f"Bus de raccordement le plus proche: {nearest_bus}")
print(f"Longueur du nouveau cable: {connection_length_km:.3f} km")
print("Parametres cable utilises:")
print(cable[["r_ohm_per_km", "x_ohm_per_km", "c_nf_per_km", "max_i_ka", "type"]])
print("Max line loading apres raccordement PV (%):", net.res_line["loading_percent"].max())


Power flow converged: True
BelalpSolar PV: 5.70 MW injectes au nouveau bus 77
Bus de raccordement le plus proche: 42
Longueur du nouveau cable: 2.303 km
Parametres cable utilises:
r_ohm_per_km    0.445
x_ohm_per_km    0.132
c_nf_per_km     190.0
max_i_ka         0.22
type               cs
Name: 45, dtype: object
Max line loading apres raccordement PV (%): 73.63037617495746


# Scenario horaire interactif

Choisir un mois, un jour et une heure. Les charges suivent un profil journalier/seasonnier simple, BelalpSolar suit un profil pvlib horaire, puis le power flow est relance. Les tailles des points et les couleurs des lignes changent avec le scenario.

In [6]:
import re
import math
import copy
import numpy as np
import pandas as pd
import pandapower as pp
import pvlib
import plotly.graph_objects as go
import plotly.io as pio
from plotly.colors import sample_colorscale
from plotly.subplots import make_subplots
from pvlib.location import Location

pio.renderers.default = "vscode"

grid_file = "/Users/jadlostan/Documents/GitHub/Swiss-PDGs/grids/pandapower_data/MV/87_0_grid.xlsx"
nodes_file = "nodes_valais.csv"
grid_id = "87_0"

pv_latitude = 46.39776
pv_longitude = 7.97738
pv_p_mw_nominal = 5.7
annual_yield_gwh_target = 12.0

# Jour et heures a comparer.
selected_day = pd.Timestamp("2025-06-21", tz="Europe/Zurich")
selected_hours = [6, 12, 18, 22]

base_net = pp.from_excel(grid_file)
nodes = pd.read_csv(nodes_file)

def id_to_str(value):
    return str(int(value)) if float(value).is_integer() else str(value)

def node_id_from_bus_name(name):
    match = re.search(r"_node_(.+)$", str(name))
    return match.group(1) if match else None

def haversine_km(lon1, lat1, lon2, lat2):
    radius_km = 6371.0
    lon1, lat1, lon2, lat2 = map(math.radians, [lon1, lat1, lon2, lat2])
    dlon = lon2 - lon1
    dlat = lat2 - lat1
    a = math.sin(dlat / 2) ** 2 + math.cos(lat1) * math.cos(lat2) * math.sin(dlon / 2) ** 2
    return 2 * radius_km * math.asin(math.sqrt(a))

nodes_grid = nodes[nodes["grid_id"] == grid_id].dropna(subset=["osmid", "longitude", "latitude"]).copy()
nodes_grid["node_id"] = nodes_grid["osmid"].apply(id_to_str)
node_lookup = nodes_grid.drop_duplicates("node_id").set_index("node_id")[["longitude", "latitude"]]

base_bus_plot = base_net.bus.copy()
base_bus_plot["bus_index"] = base_bus_plot.index
base_bus_plot["node_id"] = base_bus_plot["name"].apply(node_id_from_bus_name)
base_bus_plot = base_bus_plot.join(node_lookup, on="node_id")
base_bus_plot = base_bus_plot.dropna(subset=["longitude", "latitude"]).set_index("bus_index")

distance2 = (base_bus_plot["longitude"] - pv_longitude) ** 2 + (base_bus_plot["latitude"] - pv_latitude) ** 2
nearest_bus = int(distance2.idxmin())
nearest_lon = float(base_bus_plot.loc[nearest_bus, "longitude"])
nearest_lat = float(base_bus_plot.loc[nearest_bus, "latitude"])
connection_length_km = haversine_km(nearest_lon, nearest_lat, pv_longitude, pv_latitude)
attached_lines = base_net.line[(base_net.line["from_bus"] == nearest_bus) | (base_net.line["to_bus"] == nearest_bus)]
cable = attached_lines.iloc[0] if len(attached_lines) else base_net.line.iloc[0]

location = Location(latitude=46.39776, longitude=7.97738, tz="Europe/Zurich", altitude=2650, name="BelalpSolar")
times = pd.date_range("2025-01-01 00:00", "2025-12-31 23:00", freq="h", tz=location.tz)
solar_position = location.get_solarposition(times)
clear_sky = location.get_clearsky(times, model="ineichen")
poa = pvlib.irradiance.get_total_irradiance(
    surface_tilt=60,
    surface_azimuth=180,
    solar_zenith=solar_position["apparent_zenith"],
    solar_azimuth=solar_position["azimuth"],
    dni=clear_sky["dni"],
    ghi=clear_sky["ghi"],
    dhi=clear_sky["dhi"],
    albedo=0.35,
)
cell_temperature = pvlib.temperature.faiman(poa_global=poa["poa_global"], temp_air=5, wind_speed=2)
pdc_kw_raw = pvlib.pvsystem.pvwatts_dc(
    effective_irradiance=poa["poa_global"],
    temp_cell=cell_temperature,
    pdc0=pv_p_mw_nominal * 1000,
    gamma_pdc=-0.0035,
)
pac_kw_raw = pvlib.inverter.pvwatts(pdc=pdc_kw_raw, pdc0=pv_p_mw_nominal * 1000, eta_inv_nom=0.96).clip(lower=0)
pv_profile_mw = pac_kw_raw * (annual_yield_gwh_target * 1_000_000 / pac_kw_raw.sum()) / 1000

def load_multiplier(timestamp):
    hour = timestamp.hour + timestamp.minute / 60
    doy = timestamp.dayofyear
    morning = 0.35 * np.exp(-0.5 * ((hour - 7.5) / 2.0) ** 2)
    midday = 0.16 * np.exp(-0.5 * ((hour - 12.5) / 3.0) ** 2)
    evening = 0.50 * np.exp(-0.5 * ((hour - 19.0) / 2.5) ** 2)
    night_base = 0.55
    daily = night_base + morning + midday + evening
    seasonal = 1.0 + 0.22 * np.cos(2 * np.pi * (doy - 15) / 365.0)
    weekend = 0.95 if timestamp.weekday() >= 5 else 1.0
    return float(daily * seasonal * weekend)

def build_scenario_net(timestamp):
    net = copy.deepcopy(base_net)
    multiplier = load_multiplier(timestamp)
    net.load["p_mw"] = base_net.load["p_mw"] * multiplier
    net.load["q_mvar"] = base_net.load["q_mvar"] * multiplier
    pv_bus = pp.create_bus(net, vn_kv=float(net.bus.loc[nearest_bus, "vn_kv"]), name="bus_MV_87_0_BelalpSolar", type="n")
    pp.create_line_from_parameters(
        net,
        from_bus=nearest_bus,
        to_bus=pv_bus,
        length_km=connection_length_km,
        r_ohm_per_km=float(cable["r_ohm_per_km"]),
        x_ohm_per_km=float(cable["x_ohm_per_km"]),
        c_nf_per_km=float(cable["c_nf_per_km"]),
        max_i_ka=float(cable["max_i_ka"]),
        type=str(cable["type"]),
        name="line_87_0_to_BelalpSolar",
    )
    pv_p_mw = float(pv_profile_mw.loc[timestamp])
    pp.create_sgen(net, bus=pv_bus, p_mw=pv_p_mw, q_mvar=0.0, name="BelalpSolar PV", type="PV")
    pp.runpp(net)
    return net, pv_bus, pv_p_mw, multiplier

scenarios = []
for hour in selected_hours:
    timestamp = selected_day + pd.Timedelta(hours=hour)
    net, pv_bus, pv_p_mw, multiplier = build_scenario_net(timestamp)
    scenarios.append((timestamp, net, pv_bus, pv_p_mw, multiplier))

max_loading_global = max(float(net.res_line["loading_percent"].max()) for _, net, _, _, _ in scenarios)

fig = make_subplots(
    rows=2,
    cols=2,
    subplot_titles=[f"{ts:%H:%M} | PV {pv_p:.2f} MW | load x{mult:.2f}" for ts, _, _, pv_p, mult in scenarios],
    horizontal_spacing=0.06,
    vertical_spacing=0.10,
)

summary_rows = []
for idx, (timestamp, net, pv_bus, pv_p_mw, multiplier) in enumerate(scenarios):
    row = idx // 2 + 1
    col = idx % 2 + 1
    bus_plot = pd.concat([
        base_bus_plot,
        pd.DataFrame(
            {"longitude": [pv_longitude], "latitude": [pv_latitude], "name": ["bus_MV_87_0_BelalpSolar"], "node_id": ["BelalpSolar"]},
            index=[pv_bus],
        ),
    ])
    load_by_bus = net.load[net.load["in_service"]].groupby("bus")["p_mw"].sum()
    sgen_by_bus = net.sgen[net.sgen["in_service"]].groupby("bus")["p_mw"].sum() if len(net.sgen) else pd.Series(dtype=float)
    slack_buses = set(net.ext_grid[net.ext_grid["in_service"]]["bus"].astype(int))
    max_load = max(load_by_bus.max(), 1e-9) if len(load_by_bus) else 1

    for line_idx, line in net.line.iterrows():
        from_bus = int(line["from_bus"])
        to_bus = int(line["to_bus"])
        if from_bus not in bus_plot.index or to_bus not in bus_plot.index:
            continue
        x0, y0 = bus_plot.loc[from_bus, ["longitude", "latitude"]]
        x1, y1 = bus_plot.loc[to_bus, ["longitude", "latitude"]]
        x_mid = (x0 + x1) / 2
        y_mid = (y0 + y1) / 2
        loading = float(net.res_line.loc[line_idx, "loading_percent"])
        line_color = sample_colorscale("Viridis", loading / max(max_loading_global, 1))[0]
        fig.add_trace(
            go.Scatter(
                x=[x0, x_mid, x1],
                y=[y0, y_mid, y1],
                mode="lines",
                line=dict(color=line_color, width=4 if to_bus == pv_bus or from_bus == pv_bus else 2.2),
                hovertemplate=f"{timestamp:%H:%M}<br>edge {line_idx}<br>loading: {loading:.2f}%<extra></extra>",
                showlegend=False,
            ),
            row=row,
            col=col,
        )

    plain, loads, slacks, pvs = [], [], [], []
    for bus_idx, bus_row in bus_plot.iterrows():
        load = float(load_by_bus.get(bus_idx, 0.0))
        sgen = float(sgen_by_bus.get(bus_idx, 0.0)) if len(sgen_by_bus) else 0.0
        item = dict(bus_idx=bus_idx, lon=bus_row["longitude"], lat=bus_row["latitude"], load=load, sgen=sgen)
        if bus_idx == pv_bus:
            pvs.append(item)
        elif bus_idx in slack_buses:
            slacks.append(item)
        elif load > 0:
            loads.append(item)
        else:
            plain.append(item)

    def add_nodes(items, name, color, symbol, size_fn, line_color="white", line_width=1):
        if not items:
            return
        fig.add_trace(
            go.Scatter(
                x=[item["lon"] for item in items],
                y=[item["lat"] for item in items],
                mode="markers",
                marker=dict(symbol=symbol, size=[size_fn(item) for item in items], color=color, line=dict(color=line_color, width=line_width)),
                customdata=[[item["bus_idx"], item["load"], item["sgen"]] for item in items],
                hovertemplate="bus %{customdata[0]}<br>load: %{customdata[1]:.4f} MW<br>sgen: %{customdata[2]:.4f} MW<extra></extra>",
                name=name,
                showlegend=(idx == 0),
            ),
            row=row,
            col=col,
        )

    add_nodes(plain, "No load", "lightgray", "circle", lambda item: 5, line_color="dimgray")
    add_nodes(loads, "Load", "red", "circle", lambda item: 5 + 22 * item["load"] / max_load)
    add_nodes(slacks, "Slack / ext_grid", "gold", "star", lambda item: 24, line_color="black", line_width=2)
    add_nodes(pvs, "BelalpSolar PV", "limegreen", "diamond", lambda item: 8 + 22 * item["sgen"] / max(pv_p_mw_nominal, 1e-9), line_color="darkgreen", line_width=2)

    summary_rows.append({
        "time": timestamp.strftime("%H:%M"),
        "converged": net.converged,
        "pv_mw": pv_p_mw,
        "load_multiplier": multiplier,
        "slack_p_mw": float(net.res_ext_grid.p_mw.iloc[0]),
        "max_loading_percent": float(net.res_line.loading_percent.max()),
    })

# Une trace invisible pour afficher la colorbar commune du loading.
fig.add_trace(
    go.Scatter(
        x=[None],
        y=[None],
        mode="markers",
        marker=dict(
            colorscale="Viridis",
            cmin=0,
            cmax=max_loading_global,
            color=[0],
            colorbar=dict(title="Loading line (%)", x=1.02, y=0.5, len=0.75, thickness=18),
            showscale=True,
        ),
        hoverinfo="skip",
        showlegend=False,
    ),
    row=1,
    col=1,
)

for axis_name in fig.layout:
    if axis_name.startswith("yaxis"):
        suffix = axis_name.replace("yaxis", "")
        fig.layout[axis_name].scaleanchor = "x" + suffix
        fig.layout[axis_name].scaleratio = 1

fig.update_layout(
    title=f"Grille 87_0 avec BelalpSolar - 4 moments du {selected_day:%d.%m.%Y}",
    width=1200,
    height=950,
    dragmode="zoom",
    hovermode="closest",
    margin=dict(l=50, r=150, t=80, b=50),
    legend=dict(title="Elements", x=0.01, y=0.99, bgcolor="rgba(255,255,255,0.88)", bordercolor="rgba(0,0,0,0.25)", borderwidth=1),
)
fig.update_xaxes(title_text="Longitude")
fig.update_yaxes(title_text="Latitude")
fig.show()

display(pd.DataFrame(summary_rows))


,time,converged,pv_mw,load_multiplier,slack_p_mw,max_loading_percent
0,06:00,True,0.008685,0.631093,5.616734,25.794636
1,12:00,True,4.020066,0.567233,1.077646,52.016382
2,18:00,True,1.617312,0.792283,5.437067,25.971129
3,22:00,True,0.000000,0.604421,5.387230,24.691909


# Optimisation FFOR avec Gurobi

Cette cellule prend le power flow pandapower courant comme point de reference, linearise le reseau autour de ce point, puis resout un point de la FFOR avec Gurobi selon la direction `(alpha, beta)`.


In [ ]:
import math
import numpy as np
import pandas as pd
import pandapower as pp
import gurobipy as gp
from gurobipy import GRB

# Choix du point de power flow a optimiser.
# Si la cellule scenario horaire a ete executee, l'index 1 correspond au scenario de 12:00.
ffor_scenario_index = 1
if "scenarios" in globals() and len(scenarios):
    ffor_scenario_index = min(ffor_scenario_index, len(scenarios) - 1)
    ffor_timestamp, ffor_net, ffor_pv_bus, ffor_pv_mw, ffor_load_multiplier = scenarios[ffor_scenario_index]
else:
    ffor_timestamp, ffor_net = None, net

if not getattr(ffor_net, "converged", False):
    pp.runpp(ffor_net, calculate_voltage_angles=True)
else:
    pp.runpp(ffor_net, calculate_voltage_angles=True, init="results")

# Direction de l'objectif: changer theta_direction, ou alpha/beta directement.
theta_direction = 0.0  # rad; 0 -> minimise P_pcc, pi/2 -> minimise Q_pcc
alpha = math.cos(theta_direction)
beta = math.sin(theta_direction)

# Parametres modifiables selon les donnees disponibles.
u_min_pu = 0.95
u_max_pu = 1.05
pv_reactive_ratio = 0.33  # Qmax PV = ratio * Pmax si aucune limite explicite n'est donnee
bess_pmax_by_bus = {}     # exemple: {int(ffor_pv_bus): 1.0}
bess_qmax_by_bus = {}     # exemple: {int(ffor_pv_bus): 1.0}
bess_smax_by_bus = {}     # exemple: {int(ffor_pv_bus): 1.2}
hp_pmax_by_bus = {}       # exemple: {some_bus: 0.5}
line_limit_scale = 1.0

# Mapping pandapower -> ordre interne ppc/Ybus.
bus_lookup = ffor_net._pd2ppc_lookups["bus"]
active_buses = [int(b) for b in ffor_net.bus.index if int(b) < len(bus_lookup) and int(bus_lookup[int(b)]) >= 0]
bus_ppc_pos = {bus: int(bus_lookup[bus]) for bus in active_buses}
buses = sorted(active_buses, key=lambda bus: bus_ppc_pos[bus])
nbus = len(buses)
pos = {bus: i for i, bus in enumerate(buses)}

pcc_bus = int(ffor_net.ext_grid.loc[ffor_net.ext_grid["in_service"], "bus"].iloc[0])
pcc_pos = pos[pcc_bus]
sn_mva = float(ffor_net.sn_mva)
Ybus = ffor_net._ppc["internal"]["Ybus"].toarray()

vm_ref = np.array([float(ffor_net.res_bus.at[bus, "vm_pu"]) for bus in buses])
va_ref = np.deg2rad([float(ffor_net.res_bus.at[bus, "va_degree"]) for bus in buses])

# Garde seulement les lignes/colonnes dans l'ordre de nos variables si Ybus contient d'autres bus internes.
ppc_indices = [bus_ppc_pos[bus] for bus in buses]
Ybus = Ybus[np.ix_(ppc_indices, ppc_indices)]

def bus_power_pu(vm, va):
    voltage = vm * np.exp(1j * va)
    s_inj = voltage * np.conj(Ybus @ voltage)
    return s_inj.real, s_inj.imag

def finite_difference_bus_jacobian(vm, va, eps_vm=1e-6, eps_va=1e-6):
    jp_th = np.zeros((nbus, nbus))
    jq_th = np.zeros((nbus, nbus))
    jp_u = np.zeros((nbus, nbus))
    jq_u = np.zeros((nbus, nbus))

    for j in range(nbus):
        va_plus = va.copy(); va_plus[j] += eps_va
        va_minus = va.copy(); va_minus[j] -= eps_va
        p_plus, q_plus = bus_power_pu(vm, va_plus)
        p_minus, q_minus = bus_power_pu(vm, va_minus)
        jp_th[:, j] = (p_plus - p_minus) / (2 * eps_va)
        jq_th[:, j] = (q_plus - q_minus) / (2 * eps_va)

        vm_plus = vm.copy(); vm_plus[j] += eps_vm
        vm_minus = vm.copy(); vm_minus[j] -= eps_vm
        p_plus, q_plus = bus_power_pu(vm_plus, va)
        p_minus, q_minus = bus_power_pu(vm_minus, va)
        jp_u[:, j] = (p_plus - p_minus) / (2 * eps_vm)
        jq_u[:, j] = (q_plus - q_minus) / (2 * eps_vm)

    return jp_th * sn_mva, jp_u * sn_mva, jq_th * sn_mva, jq_u * sn_mva

p_ref_pu, q_ref_pu = bus_power_pu(vm_ref, va_ref)
p_ref_mw = p_ref_pu * sn_mva
q_ref_mvar = q_ref_pu * sn_mva
J_Ptheta, J_PU, J_Qtheta, J_QU = finite_difference_bus_jacobian(vm_ref, va_ref)

load_p = ffor_net.load[ffor_net.load["in_service"]].groupby("bus")["p_mw"].sum().reindex(buses, fill_value=0.0)
load_q = ffor_net.load[ffor_net.load["in_service"]].groupby("bus")["q_mvar"].sum().reindex(buses, fill_value=0.0)

if len(ffor_net.sgen):
    sgen_active = ffor_net.sgen[ffor_net.sgen["in_service"]].copy()
    pv_pmax = sgen_active.groupby("bus")["p_mw"].sum().reindex(buses, fill_value=0.0).clip(lower=0.0)
    pv_q_now = sgen_active.groupby("bus")["q_mvar"].sum().reindex(buses, fill_value=0.0).abs()
else:
    pv_pmax = pd.Series(0.0, index=buses)
    pv_q_now = pd.Series(0.0, index=buses)
pv_qmax = pd.Series(np.maximum(pv_q_now.values, pv_reactive_ratio * pv_pmax.values), index=buses)

bess_pmax = pd.Series({bus: float(bess_pmax_by_bus.get(bus, 0.0)) for bus in buses})
bess_qmax = pd.Series({bus: float(bess_qmax_by_bus.get(bus, 0.0)) for bus in buses})
bess_smax = pd.Series({bus: float(bess_smax_by_bus.get(bus, min(bess_pmax[bus], bess_qmax[bus]))) for bus in buses})
hp_pmax = pd.Series({bus: float(hp_pmax_by_bus.get(bus, 0.0)) for bus in buses})

pcc_p_base = float(ffor_net.res_ext_grid["p_mw"].iloc[0])
pcc_q_base = float(ffor_net.res_ext_grid["q_mvar"].iloc[0])

model = gp.Model("ffor_linearized_pf")
model.Params.OutputFlag = 0
model.Params.NumericFocus = 1

P_net = model.addVars(buses, lb=-GRB.INFINITY, name="P_net_mw")
Q_net = model.addVars(buses, lb=-GRB.INFINITY, name="Q_net_mvar")
P_pv = model.addVars(buses, lb=0.0, name="P_pv_mw")
Q_pv = model.addVars(buses, lb=-GRB.INFINITY, name="Q_pv_mvar")
P_bess = model.addVars(buses, lb=-GRB.INFINITY, name="P_bess_mw")
Q_bess = model.addVars(buses, lb=-GRB.INFINITY, name="Q_bess_mvar")
P_hp = model.addVars(buses, lb=0.0, name="P_hp_mw")
dtheta = model.addVars(buses, lb=-GRB.INFINITY, name="dtheta_rad")
du = model.addVars(buses, lb=-GRB.INFINITY, name="dU_pu")
P_pcc = model.addVar(lb=-GRB.INFINITY, name="P_pcc_mw")
Q_pcc = model.addVar(lb=-GRB.INFINITY, name="Q_pcc_mvar")

for bus in buses:
    i = pos[bus]
    model.addConstr(P_pv[bus] <= float(pv_pmax[bus]), name=f"pv_pmax[{bus}]")
    model.addConstr(Q_pv[bus] <= float(pv_qmax[bus]), name=f"pv_qmax[{bus}]")
    model.addConstr(Q_pv[bus] >= -float(pv_qmax[bus]), name=f"pv_qmin[{bus}]")

    model.addConstr(P_bess[bus] <= float(bess_pmax[bus]), name=f"bess_pmax[{bus}]")
    model.addConstr(P_bess[bus] >= -float(bess_pmax[bus]), name=f"bess_pmin[{bus}]")
    model.addConstr(Q_bess[bus] <= float(bess_qmax[bus]), name=f"bess_qmax[{bus}]")
    model.addConstr(Q_bess[bus] >= -float(bess_qmax[bus]), name=f"bess_qmin[{bus}]")
    if bess_smax[bus] > 0:
        model.addQConstr(P_bess[bus] * P_bess[bus] + Q_bess[bus] * Q_bess[bus] <= float(bess_smax[bus]) ** 2, name=f"bess_smax[{bus}]")
    else:
        model.addConstr(P_bess[bus] == 0.0, name=f"bess_p_zero[{bus}]")
        model.addConstr(Q_bess[bus] == 0.0, name=f"bess_q_zero[{bus}]")

    model.addConstr(P_hp[bus] <= float(hp_pmax[bus]), name=f"hp_pmax[{bus}]")
    model.addConstr(vm_ref[i] + du[bus] >= u_min_pu, name=f"u_min[{bus}]")
    model.addConstr(vm_ref[i] + du[bus] <= u_max_pu, name=f"u_max[{bus}]")

    pcc_term_p = P_pcc if bus == pcc_bus else 0.0
    pcc_term_q = Q_pcc if bus == pcc_bus else 0.0
    model.addConstr(P_net[bus] == P_pv[bus] + P_bess[bus] - P_hp[bus] - float(load_p[bus]) + pcc_term_p, name=f"p_balance[{bus}]")
    model.addConstr(Q_net[bus] == Q_pv[bus] + Q_bess[bus] - float(load_q[bus]) + pcc_term_q, name=f"q_balance[{bus}]")

    p_linear = p_ref_mw[i] + gp.quicksum(J_Ptheta[i, j] * dtheta[buses[j]] + J_PU[i, j] * du[buses[j]] for j in range(nbus))
    q_linear = q_ref_mvar[i] + gp.quicksum(J_Qtheta[i, j] * dtheta[buses[j]] + J_QU[i, j] * du[buses[j]] for j in range(nbus))
    model.addConstr(P_net[bus] == p_linear, name=f"lin_pf_p[{bus}]")
    model.addConstr(Q_net[bus] == q_linear, name=f"lin_pf_q[{bus}]")

model.addConstr(vm_ref[pcc_pos] + du[pcc_bus] == 1.0, name="pcc_voltage")
model.addConstr(va_ref[pcc_pos] + dtheta[pcc_bus] == 0.0, name="pcc_angle")

def line_smax_mva(line):
    from_bus = int(line["from_bus"])
    vn_kv = float(ffor_net.bus.at[from_bus, "vn_kv"])
    max_i_ka = float(line.get("max_i_ka", 0.0) or 0.0)
    parallel = float(line.get("parallel", 1.0) or 1.0)
    df = float(line.get("df", 1.0) or 1.0)
    max_loading_percent = float(line.get("max_loading_percent", 100.0) or 100.0)
    return line_limit_scale * math.sqrt(3) * vn_kv * max_i_ka * parallel * df * max_loading_percent / 100.0

# Contraintes de flux: approximation affine du flux aux deux extremites de chaque ligne.
line_flow_exprs = []
for line_idx, line in ffor_net.line[ffor_net.line["in_service"]].iterrows():
    from_bus = int(line["from_bus"])
    to_bus = int(line["to_bus"])
    if from_bus not in pos or to_bus not in pos:
        continue
    smax = line_smax_mva(line)
    if smax <= 0:
        continue

    pf0 = float(ffor_net.res_line.at[line_idx, "p_from_mw"])
    qf0 = float(ffor_net.res_line.at[line_idx, "q_from_mvar"])
    pt0 = float(ffor_net.res_line.at[line_idx, "p_to_mw"])
    qt0 = float(ffor_net.res_line.at[line_idx, "q_to_mvar"])

    # Sensibilites simples: le flux de ligne depend surtout des deux bus d'extremite.
    y_base = (float(ffor_net.bus.at[from_bus, "vn_kv"]) ** 2) / sn_mva
    r_pu = float(line["r_ohm_per_km"]) * float(line["length_km"]) / y_base
    x_pu = float(line["x_ohm_per_km"]) * float(line["length_km"]) / y_base
    if abs(r_pu + 1j * x_pu) < 1e-12:
        continue
    y_series = 1 / (r_pu + 1j * x_pu)
    b_shunt_pu = 2 * math.pi * 50.0 * float(line.get("c_nf_per_km", 0.0)) * 1e-9 * float(line["length_km"]) * y_base

    i_from = pos[from_bus]
    i_to = pos[to_bus]

    def one_line_flow(vm_from, va_from, vm_to, va_to):
        vf = vm_from * np.exp(1j * va_from)
        vt = vm_to * np.exp(1j * va_to)
        ifrom = (vf - vt) * y_series + vf * 1j * b_shunt_pu / 2
        ito = (vt - vf) * y_series + vt * 1j * b_shunt_pu / 2
        sf = vf * np.conj(ifrom) * sn_mva
        st = vt * np.conj(ito) * sn_mva
        return sf.real, sf.imag, st.real, st.imag

    eps = 1e-6
    coeffs = {}
    for local_bus, local_i in [(from_bus, i_from), (to_bus, i_to)]:
        args = [vm_ref[i_from], va_ref[i_from], vm_ref[i_to], va_ref[i_to]]
        theta_arg = 1 if local_bus == from_bus else 3
        u_arg = 0 if local_bus == from_bus else 2

        args_plus = args.copy(); args_plus[theta_arg] += eps
        args_minus = args.copy(); args_minus[theta_arg] -= eps
        flow_plus = one_line_flow(*args_plus)
        flow_minus = one_line_flow(*args_minus)
        coeffs[(local_bus, "theta")] = [(flow_plus[k] - flow_minus[k]) / (2 * eps) for k in range(4)]

        args_plus = args.copy(); args_plus[u_arg] += eps
        args_minus = args.copy(); args_minus[u_arg] -= eps
        flow_plus = one_line_flow(*args_plus)
        flow_minus = one_line_flow(*args_minus)
        coeffs[(local_bus, "u")] = [(flow_plus[k] - flow_minus[k]) / (2 * eps) for k in range(4)]

    flow_exprs = []
    for side, p0, q0, p_idx, q_idx in [("from", pf0, qf0, 0, 1), ("to", pt0, qt0, 2, 3)]:
        p_expr = p0
        q_expr = q0
        for local_bus in [from_bus, to_bus]:
            p_expr += coeffs[(local_bus, "theta")][p_idx] * dtheta[local_bus] + coeffs[(local_bus, "u")][p_idx] * du[local_bus]
            q_expr += coeffs[(local_bus, "theta")][q_idx] * dtheta[local_bus] + coeffs[(local_bus, "u")][q_idx] * du[local_bus]
        model.addQConstr(p_expr * p_expr + q_expr * q_expr <= smax ** 2, name=f"line_smax_{side}[{line_idx}]")
        line_flow_exprs.append({"line": line_idx, "side": side, "smax_mva": smax, "p_ref_mw": p0, "q_ref_mvar": q0, "p_expr": p_expr, "q_expr": q_expr})

model.setObjective(alpha * (P_pcc - pcc_p_base) + beta * (Q_pcc - pcc_q_base), GRB.MINIMIZE)
model.optimize()

if model.Status not in [GRB.OPTIMAL, GRB.SUBOPTIMAL]:
    print(f"Statut Gurobi: {model.Status}. Aucun point FFOR exploitable trouve.")
else:
    bus_solution = pd.DataFrame({
        "bus": buses,
        "P_net_mw": [P_net[bus].X for bus in buses],
        "Q_net_mvar": [Q_net[bus].X for bus in buses],
        "P_pv_mw": [P_pv[bus].X for bus in buses],
        "Q_pv_mvar": [Q_pv[bus].X for bus in buses],
        "P_bess_mw": [P_bess[bus].X for bus in buses],
        "Q_bess_mvar": [Q_bess[bus].X for bus in buses],
        "P_hp_mw": [P_hp[bus].X for bus in buses],
        "U_pu": [vm_ref[pos[bus]] + du[bus].X for bus in buses],
        "theta_deg": [np.rad2deg(va_ref[pos[bus]] + dtheta[bus].X) for bus in buses],
    })
    ffor_result = {
        "status": model.Status,
        "objective": model.ObjVal,
        "alpha": alpha,
        "beta": beta,
        "P_pcc_base_mw": pcc_p_base,
        "Q_pcc_base_mvar": pcc_q_base,
        "P_pcc_opt_mw": P_pcc.X,
        "Q_pcc_opt_mvar": Q_pcc.X,
        "delta_P_pcc_mw": P_pcc.X - pcc_p_base,
        "delta_Q_pcc_mvar": Q_pcc.X - pcc_q_base,
        "scenario_time": None if ffor_timestamp is None else str(ffor_timestamp),
    }
    line_constraints = pd.DataFrame([
        {
            "line": row["line"],
            "side": row["side"],
            "smax_mva": row["smax_mva"],
            "p_ref_mw": row["p_ref_mw"],
            "q_ref_mvar": row["q_ref_mvar"],
            "p_opt_mw": row["p_expr"].getValue(),
            "q_opt_mvar": row["q_expr"].getValue(),
            "loading_percent": 100 * math.sqrt(row["p_expr"].getValue() ** 2 + row["q_expr"].getValue() ** 2) / row["smax_mva"],
        }
        for row in line_flow_exprs
    ])

    display(pd.DataFrame([ffor_result]))
    display(bus_solution.sort_values(["P_pv_mw", "P_bess_mw", "P_hp_mw"], ascending=False).head(20))
    display(line_constraints.sort_values("loading_percent", ascending=False).head(20))
